In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import sys, os
sys.path.append(os.path.abspath(".."))
from model import EEGVAE

BACKBONE_CKPT = "/home/infres/yrothlin-24/EEG_Foundation_Model/outputs/pretraining/without_augmentation/pretraining_best.pt"
LMDB = "/projects/EEG-foundation-model/RECH202/tests_2/BCI2A/bci2a.lmdb"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def load_backbone(ckpt_path):
    config = {
        "in_channels": 19, "out_channels": 19, "adaptor_layers": 4,
        "z_channels": 64, "ch": 64, "ch_mult": [1, 2, 4, 8], "n_blocks": 2,
        "dropout": 0.0, "mode": "vq", "num_embeddings": 256, "n_quantizers": 8,
        "commitment_cost": 0.25, "decay": 0.99,
    }
    sd = torch.load(ckpt_path, map_location="cpu")["model"]
    q = [k for k in sd
         if k.startswith("autoencoder.posterior.quantize.quantizers.") and k.endswith(".embedding")]
    config["z_channels"] = sd[q[0]].shape[1]
    config["n_quantizers"] = len({int(k.split(".quantizers.")[1].split(".")[0]) for k in q})
    model = EEGVAE(**config).to(device).eval()
    model.load_state_dict(sd)
    for p in model.parameters():
        p.requires_grad = False
    return model


model = load_backbone(BACKBONE_CKPT)

In [ ]:
from pathlib import Path
from eeg_preprocessing.loaders import CustomDataset, get_subject_ids
from torch.utils.data import DataLoader


@torch.no_grad()
def extract_codebook(model, lmdb_path, device="cuda", channel_mode="mapped"):
    model.eval()
    embs, labels, subjects = [], [], []
    for sid in get_subject_ids(Path(lmdb_path)):
        ds = CustomDataset(lmdb_path, subject_ids=[sid], channel_mode=channel_mode)
        loader = DataLoader(ds, batch_size=256, num_workers=4)
        for x, y in loader:
            z = model.encode(x.to(device).float()).mode().flatten(1)
            embs.append(z.cpu().numpy())
            labels.append(y.numpy())
            subjects.append(np.full(len(y), sid))
    return np.concatenate(embs), np.concatenate(labels), np.concatenate(subjects)


emb, y, subj = extract_codebook(model, LMDB)

In [ ]:
E = emb / np.linalg.norm(emb, axis=1, keepdims=True)
S = E @ E.T
iu = np.triu_indices(len(E), k=1)
cos = S[iu]
same = y[iu[0]] == y[iu[1]]
intra, inter = cos[same], cos[~same]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(inter, bins=100, density=True, alpha=0.6, label="inter-class", color="#1f77b4")
axes[0].hist(intra, bins=100, density=True, alpha=0.6, label="intra-class", color="#d62728")
for d, c in [(intra, "#d62728"), (inter, "#1f77b4")]:
    axes[0].axvline(d.mean(), color=c, linestyle="--", lw=1)
    axes[0].axvline(np.median(d), color=c, linestyle=":", lw=1)
axes[0].set_xlabel("cosine similarity")
axes[0].set_ylabel("density")
axes[0].set_title("pairwise cosine similarity")
axes[0].legend()

classes = np.unique(y)
cents = np.stack([E[y == c].mean(0) for c in classes])
cents = cents / np.linalg.norm(cents, axis=1, keepdims=True)
C = cents @ cents.T
im = axes[1].imshow(C, vmin=-1, vmax=1, cmap="coolwarm")
axes[1].set_xticks(range(len(classes)))
axes[1].set_xticklabels(classes)
axes[1].set_yticks(range(len(classes)))
axes[1].set_yticklabels(classes)
axes[1].set_title("class centroid cosine")
fig.colorbar(im, ax=axes[1], fraction=0.046)

plt.tight_layout()
plt.show()